# Sun et al. supplementary analyses

This notebook reproduces the analyses reported in Supplementary Figure S2A–C.

- **Supplementary Figure S2A:** Compare ALDEx and the minimally modified subregion level Sun et al. Spearman analysis with the published trajectory classifications. Summarize the proportions of increasing and decreasing genes identified by ALDEx, the modified Spearman analysis, and the published trajectory analysis. These percentages support values reported in the text.
- **Supplementary Figure S2B-C:** Compare GO Biological Process results from ALDEx, the modified Spearman analysis, and the published spatial-aging-clock analysis using three-way precision and Venn diagrams.

The first section is a minimally modified version of the published Sun et al. analysis, extended from cell-type-level to cell type × subregion results. The scientific logic is retained as written in the original analysis.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr
from venn import venn

from scale_aware_st import RepositoryConfig, load_pooled_aldex_spec


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
config = RepositoryConfig.from_env()
ALDEX_RESULT_MODE = "published"  # choose "published" or "recompute"
PROJECT_DIR = config.root
DEG_DIR = config.results_dir / "DEG"
FIGURE_DIR = config.results_dir / "figures"
VENN_DIR = config.results_dir / "methods_images"

SUN_H5AD = config.external_data_dir / "sun_et_al" / "aging_coronal.h5ad"

SUN_SUBREGION_SPEARMAN = (
    DEG_DIR / "celltype_subregion_dgea_spearman_combinedbulked.csv"
)
SUN_PUBLISHED_TRAJECTORIES = (
    config.external_data_dir / "sun_et_al"
    / "2023-12-22736D-TableS7_GeneClassificationTrajectory.xlsx"
)
ALDEX_SUBREGION_RESULTS = (
    DEG_DIR / "aldex_cs_ct_subregion_cont_results_ALL.xlsx"
)

SUN_GO_AGING = (config.external_data_dir / "sun_et_al" / "2023-12-22736D-TableS8_GOBPAging.xlsx")
SUN_GO_CLOCK = (config.external_data_dir / "sun_et_al" / "2023-12-22736D-TableS11_GOBPClockGenes.xlsx")
ALDEX_GO = DEG_DIR / "aldex_go.xlsx"

for directory in (DEG_DIR, FIGURE_DIR, VENN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## 2. Minimally modified Sun et al. subregion analysis

The following analysis is adapted from the published Sun et al. code. It retains:

- normalization to a target sum of 250;
- log transformation without z-scoring;
- cell type × subregion analysis;
- exclusion of groups with fewer than 50 cells;
- age-level mean expression before Spearman correlation;
- Fisher-transformation confidence intervals;
- the published marker-exclusion list;
- the original inner-merge behavior when combining results.

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    sun = sc.read_h5ad(SUN_H5AD)
    
    import kimlabspatial.differential_expression as de
    
    CONTINUOUS_SUBREGION_INPUT_DIR = (
        DEG_DIR / "cross_study_cont_subregion"
    )
    CONTINUOUS_SUBREGION_INPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )
    
    for subregion in sun.obs["subregion"].cat.categories:
        subregion_data = sun[
            sun.obs["subregion"] == subregion
        ].copy()
    
        safe_subregion = (
            str(subregion)
            .replace("/", "-")
            .replace("_", "-")
        )
    
        de.prep_for_aldex2(
            subregion_data,
            str(CONTINUOUS_SUBREGION_INPUT_DIR),
            f"cell_type_{safe_subregion}",
            obs_key="celltype",
            count_layer=False,
        )
    
    
    def normalize_adata(adata, zscore=True):
        """Normalize to 250 transcripts, log-transform, and optionally z-score."""
        sc.pp.normalize_total(adata, target_sum=250)
        sc.pp.log1p(adata)
    
        if zscore is True:
            sc.pp.scale(adata, max_value=10)
    
        return adata
    
    
    def correlation_confidence_interval(r, n, ci=0.95):
        """Calculate Fisher-transformation confidence limits for a correlation."""
        import math
        from scipy.stats import norm
    
        z = norm.ppf(1 - (1 - ci) / 2)
        lower = math.tanh(math.atanh(r) - z / np.sqrt(n - 3))
        upper = math.tanh(math.atanh(r) + z / np.sqrt(n - 3))
    
        return lower, upper
    
    
    sun = normalize_adata(sun, zscore=False)
else:
    sun = None
    print("Published mode: loading deposited Sun pseudobulk results; skipping raw-data recomputation.")


In [ ]:
# Marker genes excluded in the published Sun et al. analysis.
exclude_markers = [
    "Gfap", "Crym", "Drd2", "Nr4a2", "Ighm", "Slc17a7", "Aldoc",
    "Adora2a", "Cd4", "C1ql3", "Stmn2", "Pvalb", "Thbs4", "Gja1",
    "Atp1a2", "C4b", "Drd1", "Lamp5", "Slc1a2", "Sparc", "Map1lc3a",
    "Tox", "Penk", "Gad2", "Chat", "Apoe", "Aqp4", "Sulf2", "Sox9",
    "Clu", "Tubb3", "Slc32a1", "Aldh1l1", "Spock2", "Nfic", "Olig1",
    "Flt1", "Pbx3", "Pdgfra", "Adamts3", "Tac1", "Cdh2", "Slc1a3",
    "Agpat3", "Fgfr3", "Msmo1", "Ntm", "Efnb2", "Apod", "Cd47",
    "Gad1", "Cdk5r1", "Cfl1", "Jak1", "Sst", "Sox2", "Dpp6", "Stub1",
    "Igf2", "Elovl5", "Fads2", "Trim2", "Syt11", "C1qa", "Npy", "Htt",
    "Pcsk1n", "Akt1", "Csf1r", "Igf1r", "Sox11", "Slc17a6", "Mtor",
    "C1qb", "Sod2", "Btg2", "Gpm6b", "Vcam1", "Nr2e1", "Parp1",
]

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    # Differential gene-expression analysis by cell type × subregion.
    dgea_dict = {}
    
    for (celltype, subregion), _ in sun.obs.groupby(
        ["celltype", "subregion"],
        observed=True,
    ):
        subset = sun[
            (sun.obs["celltype"] == celltype)
            & (sun.obs["subregion"] == subregion)
        ]
    
        if subset.n_obs < 50:
            continue
    
        key_name = f"{celltype}_{subregion}"
        dgea_dict[key_name] = {
            "gene": [],
            "spearman": [],
            "p-value": [],
            "ci_95_low": [],
            "ci_95_high": [],
        }
    
        for gene in subset.var_names:
            expression = subset[:, gene].X.flatten()
            ages = subset.obs["age"].values
    
            # Group and sample bulk by age.
            expression_by_age = pd.DataFrame(
                {"exp": expression, "age": ages}
            )
            expression_by_age = (
                expression_by_age
                .groupby("age", as_index=False)
                .agg({"exp": "mean"})
            )
    
            expression = expression_by_age["exp"].values
            ages = expression_by_age["age"].values
    
            correlation = spearmanr(ages, expression)
            ci95 = correlation_confidence_interval(
                correlation[0],
                len(ages),
                ci=0.95,
            )
    
            dgea_dict[key_name]["gene"].append(gene)
            dgea_dict[key_name]["spearman"].append(correlation[0])
            dgea_dict[key_name]["p-value"].append(correlation[1])
            dgea_dict[key_name]["ci_95_low"].append(ci95[0])
            dgea_dict[key_name]["ci_95_high"].append(ci95[1])


In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    # Remove the same marker genes excluded by the published analysis.
    for key in dgea_dict:
        remove = pd.Series(dgea_dict[key]["gene"]).isin(exclude_markers).values
    
        for field in dgea_dict[key]:
            values = np.array(dgea_dict[key][field])[~remove]
            dgea_dict[key][field] = list(values)


In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    # Combine results using the original inner-merge behavior.
    for result_number, key in enumerate(dgea_dict):
        key_df = pd.DataFrame()
        key_df["gene"] = dgea_dict[key]["gene"]
        key_df[f"{key}_PValue"] = dgea_dict[key]["p-value"]
        key_df[f"{key}_Spearman"] = dgea_dict[key]["spearman"]
        key_df[f"{key}_Lower95CI"] = dgea_dict[key]["ci_95_low"]
        key_df[f"{key}_Upper95CI"] = dgea_dict[key]["ci_95_high"]
    
        if result_number == 0:
            combined_df = key_df
        else:
            combined_df = pd.merge(
                combined_df,
                key_df,
                on="gene",
            )
    
    combined_df.to_csv(
        SUN_SUBREGION_SPEARMAN,
        index=False,
    )
    
    print(f"Saved: {SUN_SUBREGION_SPEARMAN}")
    print(f"Combined dimensions: {combined_df.shape}")


## 3. Load and classify the modified Sun et al. results

In [ ]:
sun_results = (
    pd.read_excel(
        config.additional_data_dir / "Additional_File_5.xlsx",
        sheet_name="Cont_age_ctsubregion_pseudobulk",
    )
    if ALDEX_RESULT_MODE == "published"
    else pd.read_csv(SUN_SUBREGION_SPEARMAN)
)

sun_result_dict = {
    column.rsplit("_", 1)[0]
    .replace("/", "-")
    .replace("_", "-"): None
    for column in sun_results.columns[1:]
}

for key in sun_result_dict:
    selected_columns = ["gene"] + [
        column
        for column in sun_results.columns
        if key in column.replace("/", "-").replace("_", "-")
    ]

    subset = sun_results.loc[:, selected_columns].copy()

    increasing = subset.loc[
        (subset.iloc[:, 2] > 0.3)
        & (subset.iloc[:, 3] > 0),
        "gene",
    ].values.tolist()

    decreasing = subset.loc[
        (subset.iloc[:, 2] < -0.3)
        & (subset.iloc[:, 4] < 0),
        "gene",
    ].values.tolist()

    increasing_dict = {
        gene: "Increasing"
        for gene in increasing
    }
    decreasing_dict = {
        gene: "Decreasing"
        for gene in decreasing
    }

    sun_result_dict[key] = increasing_dict | decreasing_dict

## 4. Load the published trajectory classifications

In [ ]:
trajectory = pd.read_excel(SUN_PUBLISHED_TRAJECTORIES)

trajectory_dict = {
    column
    .replace("/", "-")
    .replace("_", "-")
    .replace(" ", "-"): None
    for column in trajectory.columns[73:]
}

# Retain the first gene column and columns 73 onward exactly as in the
# original analysis.
trajectory = pd.concat(
    [trajectory.iloc[:, 0], trajectory.iloc[:, 73:]],
    axis=1,
)

trajectory_to_binary = {
    "Increasing": [
        "Increasing Gradual",
        "Increasing Late",
    ],
    "Decreasing": [
        "Decreasing Gradual",
        "Decreasing Early",
        "Midlife Decrease",
    ],
}

trajectory_lookup = {
    detailed_label: binary_label
    for binary_label, detailed_labels in trajectory_to_binary.items()
    for detailed_label in detailed_labels
}

for key in trajectory_dict:
    selected_columns = ["Gene"] + [
        column
        for column in trajectory.columns
        if key in (
            column
            .replace("/", "-")
            .replace("_", "-")
            .replace(" ", "-")
        )
    ]

    subset = trajectory.loc[:, selected_columns].copy()
    subset["simple"] = subset[selected_columns[1]].map(trajectory_lookup)
    subset = subset[~pd.isna(subset["simple"])]

    trajectory_dict[key] = dict(
        zip(subset["Gene"], subset["simple"])
    )

## 5. (Optionally postprocess and) Load ALDEx results

The ALDEx workbook is generated and postprocessed separately before this notebook is run. The existing worksheet-name conversion is retained exactly because it maps the ALDEx sheet names to the cell type × subregion naming convention used by the other two methods.

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    from glob import glob
    import kimlabspatial.differential_expression as de

    continuous_subregion_raw_files = sorted(
        glob(str(DEG_DIR / "cross_study_cont_subregion_results" / "*.xlsx"))
    )
    de.postprocess_aldex2(
        aldex_results=continuous_subregion_raw_files,
        output_dir=DEG_DIR,
        filename_prefix="cross_study_ct_",
        filename_suffix="_cont_it_results_05042026",
        output_stem="aldex_cs_ct_subregion_cont_results_ALL",
        experimental_column="age_scaled",
        pivot_filename=None,
        significant=False,
        make_pivot=False,
        signed=False,
    )
    aldex_tables = {
        sheet: pd.read_excel(ALDEX_SUBREGION_RESULTS, sheet_name=sheet)
        for sheet in pd.ExcelFile(ALDEX_SUBREGION_RESULTS).sheet_names
    }
elif ALDEX_RESULT_MODE == "published":
    aldex_tables = load_pooled_aldex_spec(
        config.additional_data_dir,
        "sun_continuous_celltype_subregion",
    )
    print(f"Loaded {len(aldex_tables)} pooled continuous-age subregion strata.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")


def comparison_key(stratum):
    """Convert region_celltype labels to the published comparison-key format."""
    region, cell_type = stratum.rsplit("_", 1)
    cell_type = cell_type.upper() if len(cell_type) <= 4 else cell_type.capitalize()
    return f"{cell_type}-{region}"


aldex_result_dict = {}
for stratum, table in aldex_tables.items():
    table = table.drop(columns=["Unnamed: 0"], errors="ignore")
    increasing = table.loc[
        (table["age_scaled:est"] > 0)
        & (table["age_scaled:pval.adj"] < 0.05),
        "gene",
    ].tolist()
    decreasing = table.loc[
        (table["age_scaled:est"] < 0)
        & (table["age_scaled:pval.adj"] < 0.05),
        "gene",
    ].tolist()
    aldex_result_dict[comparison_key(stratum)] = (
        {gene: "Increasing" for gene in increasing}
        | {gene: "Decreasing" for gene in decreasing}
    )


In [ ]:
# Harmonize result keys for comparison.
trajectory_dict = {
    key.upper(): value
    for key, value in trajectory_dict.items()
}
aldex_result_dict = {
    key.upper(): value
    for key, value in aldex_result_dict.items()
}
sun_result_dict = {
    key.upper(): value
    for key, value in sun_result_dict.items()
}

## 6. Supplementary Figure S2A: trajectory-classification agreement

In [ ]:
def agreement_score(method_dict, reference_dict):
    """
    Calculate directional agreement among genes classified by both methods.

    The denominator is the number of overlapping significant/classified genes,
    matching the original analysis.
    """
    common_genes = set(method_dict) & set(reference_dict)

    if len(common_genes) == 0:
        return np.nan, np.nan

    matches = sum(
        method_dict[gene] == reference_dict[gene]
        for gene in common_genes
    )

    return matches / len(common_genes), len(common_genes)

In [ ]:
agreement_results = []

for key in trajectory_dict:
    if key not in sun_result_dict:
        continue

    trajectory_key = trajectory_dict[key]
    sun_key = sun_result_dict[key]
    aldex_key = aldex_result_dict.get(key)

    sun_agreement, sun_n = agreement_score(
        sun_key,
        trajectory_key,
    )

    aldex_agreement = np.nan
    aldex_n = np.nan

    if aldex_key is not None:
        aldex_agreement, aldex_n = agreement_score(
            aldex_key,
            trajectory_key,
        )

    agreement_results.append(
        {
            "ct_region": key,
            "sun": sun_agreement,
            "aldex": aldex_agreement,
            "sun_n": sun_n,
            "aldex_n": aldex_n,
        }
    )

agreement_df = pd.DataFrame(agreement_results).dropna()
agreement_df["delta"] = (
    agreement_df["aldex"] - agreement_df["sun"]
)

agreement_df.to_excel(
    DEG_DIR / "sun_subregion_trajectory_agreement.xlsx",
    index=False,
)

display(agreement_df)

In [ ]:
plot_df = (
    agreement_df
    .sort_values("ct_region", ascending=False)
    .copy()
)

y_positions = np.arange(len(plot_df))

sun_values = plot_df["sun"].values.astype(float).copy()
aldex_values = plot_df["aldex"].values.astype(float).copy()

# Jitter only identical values.
same_mask = np.isclose(sun_values, aldex_values)
sun_values[same_mask] -= 0.005
aldex_values[same_mask] += 0.005

figure, axis = plt.subplots(figsize=(7, 10))

for index in range(len(plot_df)):
    axis.plot(
        [sun_values[index], aldex_values[index]],
        [index, index],
        color="grey",
        linewidth=1.5,
        alpha=0.7,
    )

axis.scatter(
    sun_values,
    y_positions,
    s=50,
    label="Spearman",
    alpha=0.9,
)
axis.scatter(
    aldex_values,
    y_positions,
    s=50,
    label="ALDEx",
    alpha=0.9,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(plot_df["ct_region"])
axis.set_xlabel("Trajectory agreement score")
axis.set_ylabel("Cell type × region")
axis.set_title(
    "ALDEx vs Spearman agreement with lifespan trajectory classifications"
)
axis.legend(frameon=False)
axis.set_xlim(0.7, 1.01)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "supp_fig_s2a_trajectory_classification_agreement.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

## 7. Supplementary Figure S2A support: increasing/decreasing proportions

In [ ]:
def direction_ratio_table(
    result_dict,
    *,
    preserve_aldex_empty_handling=False,
):
    """Calculate increasing and decreasing counts and percentages."""
    ratios = pd.DataFrame(
        index=result_dict.keys(),
        columns=[
            "Increasing_No",
            "Increasing_%",
            "Decreasing_No",
            "Decreasing_%",
        ],
    )

    for key, value in result_dict.items():
        labels = list(value.values())

        n_increasing = labels.count("Increasing")
        n_decreasing = labels.count("Decreasing")
        n_all = len(labels)

        ratios.loc[key, "Increasing_No"] = n_increasing
        ratios.loc[key, "Decreasing_No"] = n_decreasing

        if preserve_aldex_empty_handling:
            # Retain the original ALDEx-specific branching exactly.
            if n_increasing > 0 and n_decreasing > 0:
                ratios.loc[key, "Increasing_%"] = (
                    n_increasing / n_all * 100
                )
                ratios.loc[key, "Decreasing_%"] = (
                    n_decreasing / n_all * 100
                )
            elif n_decreasing > 0:
                ratios.loc[key, "Increasing_%"] = 0
                ratios.loc[key, "Decreasing_%"] = 100
            elif n_increasing > 0:
                ratios.loc[key, "Increasing_%"] = 100
                ratios.loc[key, "Decreasing_%"] = 0
            else:
                ratios = ratios.drop(key)
        else:
            ratios.loc[key, "Increasing_%"] = (
                n_increasing / n_all * 100
            )
            ratios.loc[key, "Decreasing_%"] = (
                n_decreasing / n_all * 100
            )

    return ratios

In [ ]:
aldex_ratios = direction_ratio_table(
    aldex_result_dict,
    preserve_aldex_empty_handling=True,
)
sun_ratios = direction_ratio_table(
    sun_result_dict,
)
trajectory_ratios = direction_ratio_table(
    trajectory_dict,
)

aldex_ratios.to_excel(
    DEG_DIR / "aldex_increasing_decreasing_ratios.xlsx"
)
sun_ratios.to_excel(
    DEG_DIR / "original_spearman_increasing_decreasing_ratios.xlsx"
)
trajectory_ratios.to_excel(
    DEG_DIR / "trajectory_increasing_decreasing_ratios.xlsx"
)

display(aldex_ratios)
display(sun_ratios)
display(trajectory_ratios)

In [ ]:
ratio_summary = pd.DataFrame(
    {
        "Method": [
            "ALDEx",
            "Modified Spearman",
            "Published trajectories",
        ],
        "Mean increasing (%)": [
            pd.to_numeric(
                aldex_ratios["Increasing_%"]
            ).mean(),
            pd.to_numeric(
                sun_ratios["Increasing_%"]
            ).mean(),
            pd.to_numeric(
                trajectory_ratios["Increasing_%"]
            ).mean(),
        ],
        "Mean decreasing (%)": [
            pd.to_numeric(
                aldex_ratios["Decreasing_%"]
            ).mean(),
            pd.to_numeric(
                sun_ratios["Decreasing_%"]
            ).mean(),
            pd.to_numeric(
                trajectory_ratios["Decreasing_%"]
            ).mean(),
        ],
    }
)

ratio_summary.to_excel(
    DEG_DIR / "increasing_decreasing_ratio_summary.xlsx",
    index=False,
)

display(ratio_summary)

## 8. Supplementary Figure S2B: GO Biological Process comparison

The published Spearman and spatial-clock GO results are loaded from the Sun et al. supplementary tables. The ALDEx GO workbook is generated separately in R using a minimally modified version of the published GO-analysis code.

In [ ]:
def load_go_workbook(
    workbook,
    *,
    term_column="Term",
):
    """Load one GO-term list per workbook sheet."""
    output = {}

    for sheet_name in pd.ExcelFile(workbook).sheet_names:
        table = pd.read_excel(
            workbook,
            sheet_name=sheet_name,
        )
        output[sheet_name.lower()] = (
            table[term_column].values.tolist()
        )

    return output


spearman_go = load_go_workbook(SUN_GO_AGING)
clock_go = load_go_workbook(SUN_GO_CLOCK)

if ALDEX_RESULT_MODE == "published":
    aldex_go_table = pd.read_excel(
        config.additional_data_dir / "Additional_File_5.xlsx",
        sheet_name="Cont_region_ALDEx_GO-BP",
    )
    aldex_go = {
        str(label).lower(): group["Term"].dropna().astype(str).tolist()
        for label, group in aldex_go_table.groupby("Cell type", sort=False)
    }
else:
    aldex_go = load_go_workbook(ALDEX_GO)

In [ ]:
def threeway_precision(method_terms, other1, other2):
    """
    Calculate the fraction of a method's terms in the three-way intersection.
    """
    method_terms = set(method_terms)
    other1 = set(other1)
    other2 = set(other2)

    if len(method_terms) == 0:
        return np.nan

    consensus = method_terms & other1 & other2
    return len(consensus) / len(method_terms)

In [ ]:
threeway_results = []

common_keys = (
    set(spearman_go)
    & set(aldex_go)
    & set(clock_go)
)

for key in common_keys:
    spearman_terms = spearman_go[key]
    aldex_terms = aldex_go[key]
    clock_terms = clock_go[key]

    threeway_results.append(
        {
            "group": key,
            "aldex_3way_precision": threeway_precision(
                aldex_terms,
                spearman_terms,
                clock_terms,
            ),
            "spearman_3way_precision": threeway_precision(
                spearman_terms,
                aldex_terms,
                clock_terms,
            ),
            "n_aldex": len(set(aldex_terms)),
            "n_spearman": len(set(spearman_terms)),
            "n_clock": len(set(clock_terms)),
        }
    )

threeway_df = pd.DataFrame(threeway_results)

# Retain the original filtering exactly.
threeway_df = threeway_df[
    (threeway_df["n_aldex"] > 10)
    & (threeway_df["n_spearman"] > 10)
    & (threeway_df["spearman_3way_precision"] > 0)
    & (threeway_df["aldex_3way_precision"] > 0)
].copy()

threeway_df.to_excel(
    DEG_DIR / "sun_go_threeway_precision.xlsx",
    index=False,
)

display(threeway_df)

In [ ]:
plot_df = (
    threeway_df
    .sort_values("group", ascending=False)
    .copy()
)

y_positions = np.arange(len(plot_df))

spearman_values = (
    plot_df["spearman_3way_precision"]
    .values
    .astype(float)
    .copy()
)
aldex_values = (
    plot_df["aldex_3way_precision"]
    .values
    .astype(float)
    .copy()
)

# Jitter only identical values.
same_mask = np.isclose(
    spearman_values,
    aldex_values,
)
spearman_values[same_mask] -= 0.005
aldex_values[same_mask] += 0.005

figure, axis = plt.subplots(figsize=(7, 3))

for index in range(len(plot_df)):
    axis.plot(
        [spearman_values[index], aldex_values[index]],
        [index, index],
        color="grey",
        linewidth=1.5,
        alpha=0.7,
    )

axis.scatter(
    spearman_values,
    y_positions,
    s=60,
    label="Spearman",
    alpha=0.9,
)
axis.scatter(
    aldex_values,
    y_positions,
    s=60,
    label="ALDEx",
    alpha=0.9,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(plot_df["group"])
axis.set_xlabel("3-way consensus precision")
axis.set_ylabel("Cell type")
axis.set_title(
    "Consensus overlap with published aging programs"
)
axis.legend(frameon=False)

x_maximum = np.nanmax(
    [
        plot_df["spearman_3way_precision"].max(),
        plot_df["aldex_3way_precision"].max(),
    ]
) * 1.05
axis.set_xlim(0, x_maximum)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "supp_fig_s2c_go_threeway_precision.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

## 9. Supplementary Figure S2C: Individual three-way Venn diagrams

In [ ]:
selected_cell_types = threeway_df["group"].values.tolist()

for cell_type in selected_cell_types:
    sets = {
        "ALDEx": set(aldex_go[cell_type]),
        "Spearman": set(spearman_go[cell_type]),
        "Spatial Clocks": set(clock_go[cell_type]),
    }

    venn(sets, fontsize=25)

    axis = plt.gca()
    legend = axis.get_legend()
    if legend:
        legend.remove()

    plt.savefig(
        VENN_DIR / f"threewaycomp_{cell_type}.png",
        dpi=600,
        bbox_inches="tight",
    )
    plt.show()